In [2]:
import torch
import torch.nn as nn
import random
import torch.nn.functional as F

In [4]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hid_dim = hidden_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, embed_dim)

        self.rnn = nn.LSTM(embed_dim, self.hid_dim, n_layers, dropout=dropout)
        self.dropout =nn.Dropout(dropout)
    
    def forward(self, src):
        embedded_src = self.dropout(self.embedding(src))

        out, (hidden, cell) = self.rnn(embedded_src)

        return out, hidden, cell

In [5]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embed_dim, enc_hid_dim, hidden_dim, n_layers, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.hid_dim = hidden_dim
        self.n_layers = n_layers
        self.attention = attention
        
        self.embedding = nn.Embedding(output_dim, embed_dim)

        self.rnn = nn.LSTM(embed_dim + enc_hid_dim, self.hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(self.hid_dim + enc_hid_dim + embed_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell, enc_output):
        input = input.unsqueeze(0)

        embedded_input = self.dropout(self.embedding(input))

        a = self.atention(hidden[-1:], enc_output)

        a = a.unsqueeze(1)
        enc_output = enc_output.permute(1, 0, 2)

        weight_context = torch.bmm(a, enc_output)

        weight_context = weight_context.permute(1, 0, 2)

        rnn_input = torch.cat((embedded_input, weight_context), dim = 2)

        output, (hidden, cell) = self.rnn(rnn_input, hidden, cell)

        embedded_input = embedded_input.squeeze(0)
        output = output.squeeze(0)
        weight_context = weight_context.squeeze(0)
        preds = self.fc_out((output, weight_context, embedded_input), dim=1)

        return preds, hidden, cell

In [6]:

class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init()
        
        self.attn = nn.Linear(enc_hid_dim + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, enc_outputs):
        batch_size = enc_outputs.shape[1]
        src_len = enc_outputs.shape[0]

        hidden = hidden.squeeze(0).unsqueeze(1).repeat(1, src_len, 1)

        enc_outputs = enc_outputs.permute(1, 0, 2)

        en = torch.tanh(self.attn(torch.cat((hidden, enc_outputs), dim=2)))


        attention = self.v(en).squeeze(2)

        return F.softmax(attention, dim=1)